# Sequence-to-Sequence

solving seq2seq problem using encoder-decoder model...

## Convert “21st October 2025” → “2025-10-21”

### Import libraries

In [70]:
import numpy as np
import tensorflow as tf
import random

### Build Vocabulary

In [71]:
# Character vocabularies
input_chars = list("0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ ,")
output_chars = list("0123456789-")

input_vocab = {ch: i for i, ch in enumerate(input_chars)}
output_vocab = {ch: i for i, ch in enumerate(output_chars)}
inv_output_vocab = {i: ch for ch, i in output_vocab.items()}

In [72]:
INPUT_DIM = len(input_vocab)
OUTPUT_DIM = len(output_vocab)
MAX_INPUT_LEN = 30
MAX_OUTPUT_LEN = 10
HIDDEN_DIM = 64

print(INPUT_DIM, OUTPUT_DIM)

64 11


### Generate Data

In [73]:
def generate_date_pair():
    months = ["January", "February", "March", "April", "May", "June",
              "July", "August", "September", "October", "November", "December"]
    day = random.randint(1, 28)
    month = random.choice(months)
    year = random.randint(2000, 2025)

    input_str = f"{day} {month} {year}"
    output_str = f"{year:04d}-{months.index(month)+1:02d}-{day:02d}"
    
    return input_str, output_str

# example usage
a,b = generate_date_pair()
print("Input:", a)
print("Output:", b)

Input: 6 July 2000
Output: 2000-07-06


In [74]:
def encode_seq(seq, vocab, max_len):
    seq = seq[:max_len].ljust(max_len)
    return [vocab[ch] for ch in seq]

# example usage
len(encode_seq("12 January 2020", input_vocab, MAX_INPUT_LEN))

30

In [75]:
# Create full training dataset
def build_dataset(num_samples=5000):
    encoder_input_data = []
    decoder_input_data = []
    decoder_target_data = []

    for _ in range(num_samples):
        # generate input and output string pair
        in_str, out_str = generate_date_pair()

        # encode input and output strings using vocabularies and fixed lengths
        enc = encode_seq(in_str, input_vocab, MAX_INPUT_LEN)
        dec = encode_seq(out_str, output_vocab, MAX_OUTPUT_LEN)

        # append to dataset lists
        encoder_input_data.append(enc)
        decoder_target_data.append(dec)
        decoder_input_data.append([output_vocab["-"]] + dec[:-1])
        
    return (
        np.array(encoder_input_data),
        np.array(decoder_input_data),
        np.expand_dims(np.array(decoder_target_data), -1)
        )

| Timestep | Decoder Input | Decoder Target | Model's Goal                                      |
|----------|----------------|----------------|---------------------------------------------------|
| t = 0    | "-"            | "2"            | Predict "2" given "-" and encoder context         |
| t = 1    | "2"            | "0"            | Predict "0" given "2"                             |
| t = 2    | "0"            | "..."          | Predict "2" given "0"                             |
| ...      | ...            | ...            | ...                                               |

### Let's dive into the model

In [76]:
# Build dataset
X_enc, X_dec, Y_dec = build_dataset()

In [77]:
# Model architecture
encoder_inputs = tf.keras.Input(shape=(MAX_INPUT_LEN,))
enc_emb = tf.keras.layers.Embedding(INPUT_DIM, HIDDEN_DIM)(encoder_inputs)
_, state_h, state_c = tf.keras.layers.LSTM(HIDDEN_DIM, 
                                           return_state=True)(enc_emb)


decoder_inputs = tf.keras.Input(shape=(MAX_OUTPUT_LEN,))
dec_emb = tf.keras.layers.Embedding(OUTPUT_DIM, HIDDEN_DIM)(decoder_inputs)
decoder_outputs, state_h, state_c = tf.keras.layers.LSTM(HIDDEN_DIM, 
                                       return_sequences=True, 
                                       return_state=True)(dec_emb, initial_state=[state_h, state_c])

decoder_outputs = tf.keras.layers.Dense(OUTPUT_DIM, activation='softmax')(decoder_outputs)

model = tf.keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

In [78]:
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_10      │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_11      │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_10        │ (None, 30, 64)    │      4,096 │ input_layer_10[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_11        │ (None, 10, 64)    │        704 │ input_layer_11[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_10 (LSTM)      │ [(None, 64),      │     33,024 │ embedding_10[0][… │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_11 (LSTM)      │ [(None, 10, 64),  │     33,024 │ embedding_11[0][… │
│                     │ (None, 64),       │            │ lstm_10[0][1],    │
│                     │ (None, 64)]       │            │ lstm_10[0][2]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 10, 11)    │        715 │ lstm_11[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 71,563 (279.54 KB)

 Trainable params: 71,563 (279.54 KB)

 Non-trainable params: 0 (0.00 B)

In [79]:
# Train
model.fit([X_enc, X_dec], Y_dec, batch_size=64, epochs=10)

Epoch 1/10


79/79 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - loss: 1.5961
Epoch 2/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - loss: 1.0262
Epoch 3/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 0.8975
Epoch 4/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 0.7789
Epoch 5/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.6297
Epoch 6/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.5167
Epoch 7/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - loss: 0.4137
Epoch 8/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 0.3359
Epoch 9/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.2594
Epoch 10/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - loss: 0.1980


### Inference

In [80]:
model.layers

[<InputLayer name=input_layer_10, built=True>,
 <InputLayer name=input_layer_11, built=True>,
 <Embedding name=embedding_10, built=True>,
 <Embedding name=embedding_11, built=True>,
 <LSTM name=lstm_10, built=True>,
 <LSTM name=lstm_11, built=True>,
 <Dense name=dense_5, built=True>]

| Layer Index         | Layer Type   | Role in Model              |
|---------------------|--------------|-----------------------------|
| `model.layers[0]`   | InputLayer   | Encoder input               |
| `model.layers[1]`   | InputLayer   | Decoder input               |
| `model.layers[2]`   | Embedding    | Encoder embedding           |
| `model.layers[3]`   | Embedding    | Decoder embedding           |
| `model.layers[4]`   | LSTM         | Encoder LSTM                |
| `model.layers[5]`   | LSTM         | Decoder LSTM                |
| `model.layers[6]`   | Dense        | Decoder output layer        |

In [81]:
def predict_date(input_str):
    # Step 1: Encode input string
    enc_input = np.array([encode_seq(input_str, input_vocab, MAX_INPUT_LEN)])

    # Step 2: Run encoder to get initial states
    encoder_emb = model.layers[2](enc_input)  # Encoder embedding
    _, state_h, state_c = model.layers[4](encoder_emb)  # Encoder LSTM

    # Step 3: Initialize decoder input with start token ("-")
    target_seq = np.array([[output_vocab["-"]]])
    result = []

    # Step 4: Generate output sequence token-by-token
    for _ in range(MAX_OUTPUT_LEN):
        decoder_emb = model.layers[3](target_seq)  # Decoder embedding
        decoder_output, state_h, state_c = model.layers[5](decoder_emb, initial_state=[state_h, state_c])  # Decoder LSTM
        output_token = model.layers[6](decoder_output)  # Dense layer

        # Step 5: Choose highest probability token
        sampled_token_index = np.argmax(output_token[0, -1, :])
        result.append(inv_output_vocab[sampled_token_index])

        # Step 6: Update decoder input for next timestep
        target_seq = np.array([[sampled_token_index]])

    return ''.join(result)

In [82]:

print("Input:", "21 October 2025")
print("Predicted:", predict_date("21 October 2025"))

Input: 21 October 2025
Predicted: 2025-10-21
